# BO Online Experiment (General Optimizer Stack)

This notebook implements online BO using the same optimizer/spec architecture as replay:
- `OnlineBOOptimizer` (BoTorch kernel)
- `ContinuousCandidateSpace` (bounded + optional resolution snapping)
- `ServerQueueSimulator` (real queue-backed simulator with cache)
- `BCLFOptimizationSpec` (objective/metrics contract)


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import yaml

CWD = Path.cwd().resolve()
if (CWD / 'src').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'src').exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError(f'Cannot locate repo root from cwd={CWD}')

SRC = REPO_ROOT / 'src'
if str(SRC) in sys.path:
    sys.path.remove(str(SRC))
sys.path.insert(0, str(SRC))

importlib.invalidate_caches()
for mod in [m for m in list(sys.modules) if m == 'optimization' or m.startswith('optimization.') or m == 'simulation' or m.startswith('simulation.')]:
    del sys.modules[mod]

from optimization import (
    BCLFOptimizationSpec,
    ContinuousCandidateSpace,
    ExperimentRecorder,
    OptimizerBoTorch,
    ServerQueueSimulator,
    ReplaySimulator,
    BCLFDataset,
)
from simulation.result_store import FilesystemResultStore, BandpassFilterNamingStrategy


In [ ]:
# Core runtime switches
RUN_ONLINE = False  # Set True only when settings are ready.
SIMULATOR_BACKEND = 'server'  # 'server' or 'replay'
RESUME_IF_CHECKPOINT = True
RESET_CHECKPOINT = False

SEED = 0
N_INIT = 8
BUDGET = 40
POOL_SIZE = 1500
RETRAIN_EVERY = 1

# Paths
CONFIG_PATH = REPO_ROOT / 'configs' / 'filter_bnn_online_uaw_uaw.yaml'
REPLAY_DATASET_PATH = REPO_ROOT / 'data' / 'filter_bnn_online_uaw_uaw_dataset.pkl'
OUTPUT_DIR = REPO_ROOT / 'experiments' / 'bo_online_general'
CACHE_DIR = REPO_ROOT / 'data' / 'cache' / 'filter_touchstone_cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Queue + template settings (update to your environment)
QUEUE_ROOT = REPO_ROOT / 'runtime' / 'queue'
TEMPLATE_NAME = 'templates.spiral_local'
DESIGN_NAME = None
SETUP_NAME = None
SWEEP_NAME = None
UNIT_MAP = {
    'L': 'mm', 'L1': 'mm', 'L2': 'mm',
    'W': 'mm', 'W1': 'mm', 'W2': 'mm',
    'S1': 'mm', 'S2': 'mm',
}

print('repo root:', REPO_ROOT)
print('config:', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())
print('output:', OUTPUT_DIR)
print('run_online:', RUN_ONLINE)
print('simulator_backend:', SIMULATOR_BACKEND)


In [ ]:
with CONFIG_PATH.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

variable_order = list(BCLFDataset.DEFAULT_FILTER_VARIABLE_ORDER)

reg_min = list(cfg['data']['regularization']['min'])
reg_max = list(cfg['data']['regularization']['max'])
if len(reg_min) == len(variable_order) + 1:
    reg_min = reg_min[:len(variable_order)]
    reg_max = reg_max[:len(variable_order)]
if len(reg_min) != len(variable_order):
    raise ValueError(f'Regularization dimension mismatch: len(min)={len(reg_min)} variable_order={len(variable_order)}')

variable_bounds = {
    name: (float(reg_min[i]), float(reg_max[i]))
    for i, name in enumerate(variable_order)
}

spec = BCLFOptimizationSpec(
    passband_ghz=(5.2, 5.9),
    lower_stopband_ghz=(3.6, 4.8),
    upper_stopband_ghz=(6.5, 8.7),
    min_passband_s21_db=-5.0,
    max_passband_s11_db=-5.0,
    max_stopband_s21_db=-15.0,
    s11_weight=0.5,
    stopband_weight=0.5,
)

display(pd.DataFrame({
    'variable': variable_order,
    'low': [variable_bounds[k][0] for k in variable_order],
    'high': [variable_bounds[k][1] for k in variable_order],
}))


In [ ]:
def build_optimizer(seed: int) -> OptimizerBoTorch:
    run_id = f'bo_online_general_seed{seed}'
    checkpoint_path = OUTPUT_DIR / f'{run_id}.checkpoint.json'

    if RESET_CHECKPOINT and checkpoint_path.exists():
        checkpoint_path.unlink()

    candidate_space = ContinuousCandidateSpace(
        variable_order=variable_order,
        variable_bounds=variable_bounds,
        pool_size=POOL_SIZE,
        seed=seed,
        resolution=0.01,  # mm snapping for requested geometry sampling
    )

    if SIMULATOR_BACKEND == 'server':
        result_store = FilesystemResultStore(
            root_dir=CACHE_DIR,
            naming_strategy=BandpassFilterNamingStrategy(),
            port_num=2,
        )
        simulator = ServerQueueSimulator(
            queue_root=str(QUEUE_ROOT),
            variable_order=variable_order,
            unit_map=UNIT_MAP,
            template_name=TEMPLATE_NAME,
            port_num=2,
            result_store=result_store,
            design_name=DESIGN_NAME,
            setup_name=SETUP_NAME,
            sweep_name=SWEEP_NAME,
            metrics_to_extract=['max_s11_db', 'min_s21_db'],
            run_id=run_id,
            mm_resolution=0.01,
        )
    elif SIMULATOR_BACKEND == 'replay':
        replay_dataset = BCLFDataset.from_pickle(REPLAY_DATASET_PATH, freqs=None)
        simulator = ReplaySimulator(dataset=replay_dataset)
    else:
        raise ValueError(f'Unsupported SIMULATOR_BACKEND={SIMULATOR_BACKEND}. Use server or replay.')

    optimizer = OptimizerBoTorch(
        candidate_space=candidate_space,
        simulator=simulator,
        optimization_spec=spec,
        recorder=ExperimentRecorder(output_dir=OUTPUT_DIR),
        budget=BUDGET,
        n_init=N_INIT,
        retrain_every=RETRAIN_EVERY,
        seed=seed,
        run_id=run_id,
        checkpoint_path=checkpoint_path,
        resume=(RESUME_IF_CHECKPOINT and checkpoint_path.exists()),
    )
    return optimizer


In [ ]:
if RUN_ONLINE:
    optimizer = build_optimizer(SEED)
    run_result = optimizer.run()
    history_df = pd.DataFrame(run_result.history)
    print('run_id:', run_result.run_id)
    print('best_objective:', run_result.state.best_objective)
    print('json:', run_result.recorder_output.json_path)
    print('csv:', run_result.recorder_output.csv_path)
    display(history_df.tail(10))
else:
    print('RUN_ONLINE=False; set True after queue/template settings are ready.')


In [ ]:
if RUN_ONLINE:
    import matplotlib.pyplot as plt

    plot_df = history_df.copy().sort_values('iteration')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    axes[0].plot(plot_df['iteration'], plot_df['objective'], marker='o', lw=1.6)
    axes[0].set_title('Objective by Iteration')
    axes[0].set_ylabel('objective (minimize)')

    axes[1].plot(plot_df['iteration'], plot_df['best_so_far'], marker='o', lw=1.8)
    axes[1].set_title('Best-so-far (Lower Is Better)')

    for ax in axes:
        ax.set_xlabel('iteration')
        ax.grid(True, alpha=0.25)

    plt.tight_layout()
